# 06. 나만의 Agentic AI 설계·구현 워크시트

- 예상 시간: 180~240분 (설계 60분 + 구현 120~180분)
- 선수 실습: Part3 전체(0~5장), 특히 02-1(State/Node/Edge), 03-3(Agentic RAG),
  04-1(Memory), 04-2(Checkpoint/Human-in-the-Loop)
- 실습 난이도: 고급 (자유 설계)
- 핵심 기술: 지금까지 배운 State·Node·Edge, Tool 호출, RAG, Memory, Human-in-the-Loop를
  하나의 Agent로 통합 설계·구현
- 최종 산출물: 내가 정한 주제로 직접 설계하고 구현한 나만의 Agentic AI
- 버전: 학생용 실습본 (`Part4_설계_워크시트.docx`에서 설계한 내용에 맞춰 TODO를 직접
  채워야 합니다)

## 사용 안내

- 이 노트북에는 정답이 없다. 반드시 `Part4_설계_워크시트.docx`에서 요구사항 정의와
  아키텍처 설계를 먼저 마친 뒤, 그 설계를 바탕으로 이 노트북의 TODO를 채운다.
- 아래 예시(State 필드명, Node 이름, 진로/직업 정보)는 "진로상담 에이전트"를 기준으로
  안내하지만, 본인이 다른 주제를 설계했다면 이름과 내용을 자유롭게 바꿔도 된다. 다만
  State → Memory → RAG(선택) → Node → Routing → Graph → 테스트로 이어지는 순서와
  각 구성요소의 역할은 그대로 유지하는 것을 권장한다.
- 막히는 구간마다 어느 장의 개념을 참고하면 되는지 각 섹션 제목 옆에 표시해 두었다.

## 1. 이 워크시트의 목적

Part3의 각 장은 State·Node·Edge(2장), Tool 호출(1장), RAG(3장), Memory와
Human-in-the-Loop(4장), 그리고 이 모두를 하나로 합친 통합 Knowledge Agent(5장)를
개별 예제로 다뤘다. 이번 워크시트는 그 구성요소들을 **내가 직접 고른 주제**에
적용해, 요구사항 정의부터 아키텍처 설계, 실제 구현, 시나리오 테스트까지 한 번에
끝까지 가져가는 것을 목표로 한다. 예시 주제는 "진로상담 에이전트"이며, 사용자의
관심분야·강점·고민을 파악하고(Intake), 관련 진로/직업 정보를 검색하고(RAG),
과거 상담 내용을 기억하고(Memory), 최종 추천 전에 사용자 확인을 받는(HITL) 흐름을
갖는다.

## 2. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀은 지금까지의 실습과 동일한 표준 환경 설정이며 그대로 실행하면 된다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.

In [ ]:
from typing import Literal, TypedDict

from langchain_core.documents import Document
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_chat_model, get_embedding_model
from agentic_ai.notebook_utils import print_environment_summary, show_graph
from agentic_ai.paths import OUTPUT_DIR
from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store

settings = get_settings()
print_environment_summary(settings, needs_chat_model=True, needs_embedding_model=True)

chat_model = get_chat_model()


def call_llm(prompt: str) -> str:
    return chat_model.invoke(prompt).content

## 3. State 설계 (워크시트 2.1과 연결)

워크시트 2.1에서 표로 정리한 State 필드를 아래 `TypedDict`로 옮겨 적는다. 최소
다음 네 가지 범주를 포함하는 것을 권장한다.

1. 대화/입력 — 사용자와 주고받은 메시지
2. 사용자 프로필 — 여러 세션에 걸쳐 기억할 정보(Long-Term Memory로 이어짐)
3. 이번 턴에서만 쓰는 작업 데이터 — 검색 결과, 중간 판단값 등
4. 최종 응답 — Agent가 사용자에게 돌려줄 결과

In [ ]:
# 아래는 "진로상담 에이전트" 예시 필드다. 워크시트 2.1에서 본인이 설계한 필드가
# 다르다면 이름과 개수를 자유롭게 바꿔도 된다.
class CareerCounselorState(TypedDict):
    messages: list  # 대화 메시지 누적 (HumanMessage / AIMessage)
    user_id: str  # Long-Term Memory namespace를 구분하는 사용자 식별자
    user_profile: dict  # 관심분야·강점·고민 등, Intake에서 채워지는 프로필

    # TODO: 워크시트 2.1에서 추가로 설계한 State 필드를 여기에 채워 넣으세요.
    # 예) retrieved_docs: list[dict]  (검색된 진로/직업 정보)
    # 예) pending_confirmation: bool  (최종 추천 확인 대기 여부)
    # 예) final_answer: str           (사용자에게 보여줄 최종 응답)

## 4. Memory 함수 (Part3 4장 참고)

Part3 4-1장에서 만든 `save_fact()` / `search_memory()` 패턴을 이 에이전트의
사용자 프로필 저장에 그대로 적용한다. 저장 함수는 그대로 재사용할 수 있도록
완성해 두었고, 검색 함수는 본인 도메인에 맞게 다시 작성해 본다.

In [ ]:
CURRENT_TICK = 0
MEMORY_STORE: dict[str, list[dict]] = {}


def save_user_fact(user_id: str, key: str, value: str, category: str = "fact") -> dict:
    """확인된 사용자 정보를 MEMORY_STORE에 저장한다 (04-1의 save_fact와 동일한 패턴)."""
    record = {
        "key": key,
        "value": value,
        "category": category,
        "created_at": CURRENT_TICK,
    }
    MEMORY_STORE.setdefault(user_id, []).append(record)
    return record

In [ ]:
def search_user_memory(user_id: str, keyword: str) -> list[dict]:
    """key 또는 value에 keyword가 포함된 사용자 Memory 레코드를 검색해 반환한다."""
    raise NotImplementedError(
        "TODO: MEMORY_STORE.get(user_id, [])로 레코드 목록을 가져온 뒤, "
        "key 또는 value에 keyword가 포함된 레코드만 리스트로 반환하세요. "
        "(04-1_context_state_memory.ipynb의 search_memory() 참고)"
    )

## 5. 진로/직업 정보 검색 (RAG, Part3 3장 참고)

상담에 참고할 진로/직업 정보를 Chroma Vector Store에 저장해 두고, 사용자
관심분야와 관련된 항목을 검색한다. 예시 데이터 5건을 준비해 두었으니, 본인이
다루고 싶은 분야가 있다면 자유롭게 추가하거나 교체해도 된다. 이 섹션은 워크시트
2.5에서 "RAG 필요함"으로 설계한 경우에만 사용한다 — 필요 없다고 설계했다면 이
섹션 전체를 건너뛰어도 된다.

In [ ]:
SAMPLE_CAREER_DOCS = [
    {"doc_id": "c1", "field": "데이터 분석가",
     "text": "데이터 분석가는 비즈니스 문제를 데이터로 정의하고, SQL·Python·통계 지식을 "
             "활용해 인사이트를 도출한다. 커뮤니케이션 능력과 도메인 이해가 중요하다."},
    {"doc_id": "c2", "field": "AI/ML 엔지니어",
     "text": "AI/ML 엔지니어는 머신러닝 모델을 설계·훈련·배포한다. 수학·통계 기초와 함께 "
             "실제 서비스에 모델을 연결하는 엔지니어링 역량이 요구된다."},
    {"doc_id": "c3", "field": "UX 디자이너",
     "text": "UX 디자이너는 사용자 리서치를 바탕으로 제품의 사용성을 설계한다. 공감 능력과 "
             "프로토타이핑 도구 활용 능력이 핵심이다."},
    {"doc_id": "c4", "field": "직업훈련 강사",
     "text": "직업훈련 강사는 현장 실무 경험을 교육 콘텐츠로 재구성하고, 학습자 수준에 맞춰 "
             "커리큘럼을 설계·운영한다."},
    {"doc_id": "c5", "field": "프로덕트 매니저",
     "text": "프로덕트 매니저는 시장과 사용자 요구를 제품 요구사항으로 번역하고, 개발·디자인·"
             "비즈니스 이해관계자 사이의 우선순위를 조율한다."},
    # TODO(선택): 본인 워크시트 설계에 맞는 진로/직업 정보를 추가하세요.
]

embedding_model = get_embedding_model()
# 03-2·03-3 데이터와 섞이지 않도록 이 워크시트 전용 Collection을 사용한다.
career_store = get_chroma_store("career_counselor_worksheet", embedding_model=embedding_model)

career_documents = [
    Document(page_content=doc["text"], metadata={"doc_id": doc["doc_id"], "field": doc["field"]})
    for doc in SAMPLE_CAREER_DOCS
]
added, count = add_documents_if_empty(
    career_store,
    career_documents,
    ids=[doc["doc_id"] for doc in SAMPLE_CAREER_DOCS],
)
print(f"진로정보 문서 수: {count}, 새로 추가: {added}")

In [ ]:
def retrieve_career_info(query: str, k: int = 2) -> list[dict]:
    """query와 유사한 진로/직업 정보를 최대 k개 검색해 반환한다."""
    raise NotImplementedError(
        "TODO: career_store.similarity_search(query, k=k)로 검색한 뒤, 각 결과의 "
        "page_content와 metadata를 {'field': ..., 'text': ...} 형태의 dict 리스트로 "
        "변환해 반환하세요. (03-2_embedding_vector_retrieval.ipynb 참고)"
    )

## 6. Node 함수 구현 (워크시트 2.2, 2.6과 연결)

워크시트 2.2에서 설계한 Node 목록을 아래 세 가지 대표 Node를 중심으로 구현한다.
Node 이름이나 개수가 본인 설계와 다르다면 자유롭게 추가·수정한다.

### 6.1 intake_node — 사용자 발화에서 프로필 정보 추출

In [ ]:
def intake_node(state: CareerCounselorState) -> dict:
    """가장 최근 사용자 발화에서 진로상담에 필요한 정보를 추출해 user_profile에 반영한다."""
    raise NotImplementedError(
        "TODO: 1) state['messages']의 마지막 HumanMessage 내용을 가져온다. "
        "2) call_llm()에 '아래 발화에서 관심분야·강점·고민을 짧게 추출해줘' 같은 "
        "프롬프트를 보낸다. 3) 추출된 정보를 save_user_fact()로 저장한다. "
        "4) {'user_profile': {...}} 형태의 부분 update를 반환한다."
    )

### 6.2 counseling_node — 검색 결과와 기억을 활용한 상담 응답 생성

In [ ]:
def counseling_node(state: CareerCounselorState) -> dict:
    """user_profile, 검색된 진로정보, 과거 Memory를 조합해 상담 응답을 생성한다."""
    raise NotImplementedError(
        "TODO: 1) retrieve_career_info()로 user_profile과 관련된 진로정보를 검색한다 "
        "(RAG를 쓰지 않기로 설계했다면 생략). 2) search_user_memory()로 과거에 저장된 "
        "사실·선호를 조회한다. 3) 검색 결과와 Memory를 Context에 포함해 call_llm()으로 "
        "상담 응답을 생성한다. 4) {'messages': [AIMessage(content=응답)]} 형태로 "
        "반환한다."
    )

### 6.3 confirm_recommendation_node — 최종 추천 전 Human-in-the-Loop 확인 (Part3 4-2장 참고)

워크시트 2.6에서 "이 시점에는 사용자 확인이 반드시 필요하다"고 설계한 지점을
`interrupt()`로 구현한다.

In [ ]:
def confirm_recommendation_node(state: CareerCounselorState) -> dict:
    """최종 진로 추천을 확정하기 전, interrupt()로 사용자 확인을 받는다."""
    raise NotImplementedError(
        "TODO: 04-2_checkpoint_human_in_the_loop.ipynb의 interrupt() 패턴을 참고해, "
        "직전 counseling_node의 응답을 interrupt(...) payload로 전달해 실행을 멈추세요. "
        "재개 시 Command(resume=...)로 받은 사용자 응답을 반영해 최종 결과를 "
        "{'messages': [...]} 형태로 반환하세요."
    )

## 7. 라우팅 설계 (워크시트 2.2와 연결)

워크시트 2.2의 흐름도에서 조건에 따라 Node가 갈라지는 지점(Conditional Edge)을
함수로 구현한다.

In [ ]:
def route_after_counseling(state: CareerCounselorState) -> Literal["confirm", "end"]:
    """counseling_node 다음에 사용자 확인이 필요한지 판단해 다음 Node 이름을 반환한다."""
    raise NotImplementedError(
        "TODO: 워크시트 2.6에서 정한 HITL 조건(예: 응답에 특정 추천 문구가 포함되는지, "
        "또는 대화 턴 수가 일정 이상인지)에 따라 'confirm' 또는 'end'를 반환하는 "
        "라우팅 로직을 작성하세요."
    )

## 8. Graph 구성

워크시트 2.2의 Node/Edge 설계를 그대로 `StateGraph`로 옮긴다.

In [ ]:
def build_worksheet_graph():
    """워크시트 2.2에서 설계한 Node/Edge 구성대로 StateGraph를 만들고 compile해 반환한다."""
    raise NotImplementedError(
        "TODO: StateGraph(CareerCounselorState)를 만들고 add_node로 intake_node·"
        "counseling_node·confirm_recommendation_node를 등록한 뒤, add_edge(START, ...)와 "
        "add_conditional_edges('counseling', route_after_counseling, "
        "{'confirm': 'confirm_recommendation', 'end': END})로 워크시트 2.2의 흐름도를 "
        "그대로 연결하세요. 마지막에 checkpointer=MemorySaver()로 compile해 반환하세요."
    )


worksheet_app = build_worksheet_graph()

In [ ]:
# 그래프 그리기 — 위 build_worksheet_graph()를 완성한 뒤 실행하세요.
show_graph(worksheet_app)

## 9. 시나리오 테스트 (워크시트 5.1과 연결)

워크시트 5.1에 적어 둔 시나리오 중 하나를 골라 실제로 실행해 본다. `thread_id`를
바꾸면 서로 다른 상담 세션으로 취급된다.

In [ ]:
# TODO: 워크시트 5.1의 시나리오 1 사용자 입력을 그대로 옮겨 적으세요.
scenario_config = {"configurable": {"thread_id": "scenario-1"}}
scenario_input = {
    "messages": [HumanMessage(content="여기에 시나리오 1의 사용자 입력을 옮겨 적으세요.")],
    "user_id": "student-01",
    "user_profile": {},
}
scenario_result = worksheet_app.invoke(scenario_input, config=scenario_config)
print(scenario_result)

**관찰 기록** — 워크시트 5.1의 표에 아래 내용을 옮겨 적는다.

- 기대 응답:
- 실제 응답:
- 통과 여부:
- 개선할 점:

## 10. 결과 저장

In [ ]:
worksheet_log = {
    "scenario_1": {
        "input": scenario_input["messages"][0].content,
        "result": scenario_result,
    },
}
saved_path = save_log(worksheet_log, OUTPUT_DIR / "logs" / "06_my_agentic_ai_worksheet_log.json")
print("로그 저장 위치:", saved_path)

## 11. 도전 과제 (선택)

1. `user_profile`이 세션마다 초기화되지 않도록, `InMemoryStore` + namespace로 여러
   `thread_id`에 걸쳐 프로필을 공유하도록 확장해 본다 (Part3 4-1장 8.2절 참고).
2. `retrieve_career_info()`의 검색 결과가 부족할 때, 03-3장의 Query Rewrite처럼
   검색어를 바꿔 한 번 더 검색하는 조건 분기를 추가해 본다.
3. `confirm_recommendation_node`에서 사용자가 추천에 동의하지 않으면, 처음부터
   다시 묻지 않고 `counseling_node`로 되돌아가 다른 추천을 생성하도록 만들어 본다.
4. 진로상담이 아닌 다른 주제로 State·Node 이름을 바꿔 같은 구조를 재사용해 본다.

## 12. 회고

- 무엇을 배웠는가:
- 설계 단계에서 가장 어려웠던 결정과 그 이유:
- 구현하면서 설계를 바꾼 부분과 그 이유:
- 다음에 개선하고 싶은 점: